# Week 1 (First Half) - Preprocessing & Deduplication
**NLP-Based Quality Feedback Analyzer**

This notebook covers the initial phase of the Quality Feedback Analyzer project (Week 1, First Half). 
The tasks completed in this notebook include:
1. Loading and exploring the raw Amazon Product Reviews CSV.
2. Identifying the review text (`Text`) and rating (`Score`) columns.
3. Checking for a product category column (not present in this dataset).
4. Performing rigorous deduplication:
   - Dropping exact duplicates on raw review text.
   - Dropping near-duplicates by normalizing the text (lowercasing, removing punctuation/special characters, and collapsing spacing) and checking for duplicates on the normalized text.
   - Sampling down to exactly 10,000 unique reviews.
5. Performing basic cleaning (lowercasing and stripping punctuation) on the sampled reviews.
6. Exporting the stage 1 results to `data/processed/reviews_stage1.csv`.


In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import re
import string


## 1. Load Raw CSV

In [2]:
# Load the raw CSV from data/raw/
raw_data_path = '../data/raw/Reviews.csv'
df = pd.read_csv(raw_data_path)
print(f"Dataset loaded successfully with shape: {df.shape}")


Dataset loaded successfully with shape: (568454, 10)


## 2. Exploratory Data Analysis
Inspect columns, data types, missing values, ratings distribution, and sample rows.

In [3]:
# Print columns and data types
print("Dataset Columns:")
print(df.columns.tolist())
print("-" * 50)

# Check for null values
print("Missing values per column:")
print(df.isnull().sum())
print("-" * 50)

# Rating (Score) distribution
print("Rating (Score) distribution:")
score_counts = df['Score'].value_counts().sort_index()
print(score_counts)
print("-" * 50)

# Display sample rows
print("First 5 rows of the raw dataset:")
df.head(5)


Dataset Columns:
['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']
--------------------------------------------------
Missing values per column:
Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64
--------------------------------------------------
Rating (Score) distribution:
Score
1     52268
2     29769
3     42640
4     80655
5    363122
Name: count, dtype: int64
--------------------------------------------------
First 5 rows of the raw dataset:


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


## 3. Identify Review Text and Rating Columns
- **Review Text Column**: `'Text'`
- **Rating Column**: `'Score'`

## 4. Filter by Product Category (If Exists)

In [4]:
# Check if a category column exists in the dataset
category_cols = [col for col in df.columns if 'category' in col.lower()]
print(f"Detected category columns: {category_cols}")

if category_cols:
    print(f"Filtering by category column: {category_cols[0]}")
    # Example filtering if category column existed:
    # df = df[df[category_cols[0]] == 'some_category']
else:
    print("No product category column exists in this dataset. Proceeding with the full dataset.")


Detected category columns: []
No product category column exists in this dataset. Proceeding with the full dataset.


## 5. Deduplication and Sampling
We first remove exact duplicates of the review text. Then, we normalize the text to identify and remove near-duplicates. Finally, we sample down to exactly 10,000 unique rows.

In [5]:
# Record initial count
initial_count = len(df)
print(f"Initial row count: {initial_count}")

# a) Drop exact duplicate review text first
df_unique_exact = df.drop_duplicates(subset=['Text'])
exact_dedup_count = len(df_unique_exact)
exact_removed = initial_count - exact_dedup_count
print(f"Rows after dropping exact duplicates: {exact_dedup_count} (Removed {exact_removed} rows)")

# b) Check for near-duplicates using normalized/lowercased text hashing
def normalize_text_for_dedup(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation and special characters
    text = re.sub(r'\s+', ' ', text).strip()  # collapse whitespaces
    return text

print("Normalizing text for near-duplicate checking (this may take a few moments)...")
df_unique_exact = df_unique_exact.copy()
df_unique_exact['normalized_text'] = df_unique_exact['Text'].apply(normalize_text_for_dedup)

# Drop near duplicates
df_deduplicated = df_unique_exact.drop_duplicates(subset=['normalized_text'])
final_dedup_count = len(df_deduplicated)
near_removed = exact_dedup_count - final_dedup_count
total_removed = initial_count - final_dedup_count

print(f"Rows after dropping near-duplicates: {final_dedup_count} (Removed an additional {near_removed} near-duplicates)")
print(f"Total duplicates removed: {total_removed}")

# c) Sample down to exactly 10,000 unique rows after deduplication
if final_dedup_count >= 10000:
    df_sampled = df_deduplicated.sample(n=10000, random_state=42)
else:
    raise ValueError(f"Not enough unique rows ({final_dedup_count}) to sample 10,000.")

print(f"Final sampled dataset size: {len(df_sampled)} rows")


Initial row count: 568454


Rows after dropping exact duplicates: 393579 (Removed 174875 rows)
Normalizing text for near-duplicate checking (this may take a few moments)...


Rows after dropping near-duplicates: 393104 (Removed an additional 475 near-duplicates)
Total duplicates removed: 175350
Final sampled dataset size: 10000 rows


## 6. Apply Lowercasing, Remove Punctuation/Special Characters, and Save Stage 1

In [6]:
# Function to lowercase and remove punctuation and special characters
def clean_text_stage1(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply preprocessing to create 'cleaned_text' column
df_sampled = df_sampled.copy()
df_sampled['cleaned_text'] = df_sampled['Text'].apply(clean_text_stage1)

# Select required columns: original text, cleaned text, and rating (Score)
output_df = df_sampled[['Text', 'cleaned_text', 'Score']]

# Save to data/processed/reviews_stage1.csv
output_path = '../data/processed/reviews_stage1.csv'
output_df.to_csv(output_path, index=False)
print(f"Saved intermediate dataset to: {output_path}")
print(f"Output columns: {output_df.columns.tolist()}")
print(f"Output shape: {output_df.shape}")


Saved intermediate dataset to: ../data/processed/reviews_stage1.csv
Output columns: ['Text', 'cleaned_text', 'Score']
Output shape: (10000, 3)


### Summary of Data Processing and Deduplication

During the preprocessing phase for Week 1 (first half), the following steps were executed on the raw dataset:

1. **Initial Raw Dataset**: The raw CSV contains **568,454** review entries.
2. **Exact Duplicate Removal**:
   - Dropped exact duplicates on the raw review text (`Text` column).
   - Rows after exact deduplication: **393,579** (removed **174,875** exact duplicates).
3. **Near-Duplicate Removal**:
   - Normalized review text (lowercasing, punctuation/special characters removed, whitespaces collapsed) and dropped duplicates on this representation.
   - Rows after near-duplicate deduplication: **393,104** (removed an additional **475** near-duplicates).
   - Total duplicate/near-duplicate reviews removed: **175,350** reviews.
4. **Final Sampled Dataset**:
   - Randomly sampled **exactly 10,000** unique reviews from the remaining 393,104 deduplicated entries using `random_state=42` to ensure reproducible results.
   - The final processed dataset, saved as `data/processed/reviews_stage1.csv`, contains exactly **10,000** unique rows and three columns: `Text` (original review text), `cleaned_text` (lowercased and punctuation/special character stripped review text), and `Score` (original rating).
